In [1]:
def transpose(A):
    return [[A[j][i] for j in range(len(A))] for i in range(len(A[0]))]

def matmul(A, B):
    # Perkalian matriks A x B
    result = [[0 for _ in range(len(B[0]))] for _ in range(len(A))]
    for i in range(len(A)):
        for j in range(len(B[0])):
            for k in range(len(B)):
                result[i][j] += A[i][k] * B[k][j]
    return result

def pca_scratch(data, n_components):
    # 1. Standarisasi (Mean Centering)
    n_samples = len(data)
    n_features = len(data[0])

    means = [sum(col) / n_samples for col in transpose(data)]
    data_centered = []
    for row in data:
        data_centered.append([row[i] - means[i] for i in range(n_features)])

    # 2. Hitung Matriks Kovarians
    # Cov = (X^T * X) / (n-1)
    data_T = transpose(data_centered)
    covariance_matrix = matmul(data_T, data_centered)
    for i in range(len(covariance_matrix)):
        for j in range(len(covariance_matrix[0])):
            covariance_matrix[i][j] /= (n_samples - 1)

    # 3. Hitung Eigenvalues & Eigenvectors
    # Catatan: Tanpa NumPy, kita menggunakan metode Power Iteration sederhana
    # untuk mencari n_components utama.

    eigenvectors = []
    current_matrix = [row[:] for row in covariance_matrix]

    for _ in range(n_components):
        # Inisialisasi vektor awal
        v = [1.0] * n_features
        for _ in range(100): # Iterasi untuk konvergensi
            # v = Cov * v
            v_new = [sum(current_matrix[i][j] * v[j] for j in range(n_features)) for i in range(n_features)]
            # Normalisasi v
            norm = sum(x**2 for x in v_new)**0.5
            v = [x / norm for x in v_new]

        eigenvectors.append(v)

        # Deflasi matriks untuk mencari komponen berikutnya
        # Matrix = Matrix - eigenvalue * v * v^T
        eigenvalue = sum(v[i] * sum(current_matrix[i][j] * v[j] for j in range(n_features)) for i in range(n_features))
        for i in range(n_features):
            for j in range(n_features):
                current_matrix[i][j] -= eigenvalue * v[i] * v[j]

    # 4. Proyeksi Data ke Komponen Utama
    # Hasil = Data_Centered * Eigenvectors^T
    projected_data = matmul(data_centered, transpose(eigenvectors))

    return projected_data

# Contoh Penggunaan:
dataset = [
    [2.5, 2.4],
    [0.5, 0.7],
    [2.2, 2.9],
    [1.9, 2.2],
    [3.1, 3.0],
    [2.3, 2.7],
    [2.0, 1.6],
    [1.0, 1.1],
    [1.5, 1.6],
    [1.1, 0.9]
]

hasil_pca = pca_scratch(dataset, n_components=1)
print("Data setelah proyeksi PCA (1 Dimensi):")
for row in hasil_pca:
    print(row)


Data setelah proyeksi PCA (1 Dimensi):
[0.8279701862010878]
[-1.7775803252804292]
[0.9921974944148884]
[0.2742104159753993]
[1.6758014186445398]
[0.9129491031588078]
[-0.09910943749844434]
[-1.1445721637986601]
[-0.4380461367624502]
[-1.2238205550547405]
